# Librerie

In [0]:
%run ../FASE1/00_utils

In [0]:
%run ./00_utility

In [0]:
%pip install pandarallel

In [0]:
import time
import warnings
import mlflow
import mlflow.deployments
import numpy as np
import pandas as pd
from pandarallel import pandarallel
from datetime import datetime
from pyspark.sql import functions as F

pandarallel.initialize(nb_workers = 16, progress_bar=False)

# Parametri

In [0]:
catalog = get_catalog()
print("Catalog: ", catalog)
model_version = 3
model_uri = f"models:/ta_coll.whatif.whatif_model_sostituzione/{model_version}" 

# Model wrapping

## Definizione del modello "wrapper"
Il modello wrapper contiene sia le funzioni per l'estrazione e la lavorazione degli input per prepararli all'inferenza sia il modello XGBoost per la predizione.

In [0]:
class WhatIfSpostamentoModel(mlflow.pyfunc.PythonModel):
    def __init__(self):
        """
        Inizializzazione del client per chiamare i feature serving endpoints.
        """
        import mlflow.deployments
        self.client = mlflow.deployments.get_deploy_client("databricks")

    def load_context(self, context):
        """
        Caricare le configurazioni.
        """
        import json
        # Caricamento del dizionario con le configurazioni
        cfg = context.model_config or {}
        
        # Config endpoints
        self.endpoint_feature_all_slots = cfg.get("endpoint_feature_all_slots")

        # Config delle features
        self.programma_da_spostare_features_cols_to_drop = cfg.get("programma_da_spostare_features_cols_to_drop")
        self.slot_destinazione_features_cols_to_keep = cfg.get("slot_destinazione_features_cols_to_keep")
        with open(context.artifacts["ohe_json"], "r") as f:
            self.ohe_config = json.load(f)

        # Loading del modello e delle signature
        self.model = mlflow.sklearn.load_model(context.artifacts["inference_model"])
        self.model_info = mlflow.models.get_model_info(context.artifacts["inference_model"])
        self.signature = self.model_info.signature
        self.expected_columns = [col.name for col in self.signature.inputs]
        self.signature_types = {col.name: col.type.name for col in self.signature.inputs}
    
    def _fetch_features(
        self,
        programma_da_spostare: str,
        destinazione: dict,
    ):
        import pandas as pd

        # ------------------------------------------------------------------
        # Feature lookup for the program to move
        # ------------------------------------------------------------------
        df_da_spostare = pd.DataFrame(
            [{"ID": programma_da_spostare}]
        )
        resp_da_spostare = self.client.predict(
            endpoint=self.endpoint_feature_all_slots,
            inputs={
                "dataframe_records": df_da_spostare.to_dict(orient="records")
            },
        )

        programma_da_spostare_features = pd.DataFrame(
            resp_da_spostare["outputs"]
        )

        # ------------------------------------------------------------------
        # Feature lookup for all programs in the destination slot
        # ------------------------------------------------------------------
        programmi_contesto = destinazione["programmi_contesto"]

        df_destinazione = pd.DataFrame(
            {"ID": programmi_contesto}
        )

        resp_destinazione = self.client.predict(
            endpoint=self.endpoint_feature_all_slots,
            inputs={
                "dataframe_records": df_destinazione.to_dict(orient="records")
            },
        )

        slot_destinazione_features = pd.DataFrame(
            resp_destinazione["outputs"]
        )

        # ------------------------------------------------------------------
        # If multiple programs overlap the destination slot,
        # keep the one occupying the largest fraction of the slot.
        # ------------------------------------------------------------------
        if len(slot_destinazione_features) > 1:

            orario = destinazione["orario"]
            if isinstance(orario, list):
                orario = orario[0]

            h, m = map(int, orario.split(":"))
            orario_inizio_slot = h * 3600 + m * 60

            durata_slot = float(programma_da_spostare_features["durata_minuti"].iloc[0]) * 60
            orario_fine_slot = orario_inizio_slot + durata_slot

            slot_destinazione_features = slot_destinazione_features.copy()

            def calc_overlap(row):
                inizio = max(row["ORA_INIZIO_TRX"], orario_inizio_slot)
                fine = min(row["ORA_FINE_TRX"], orario_fine_slot)
                overlap = max(0.0, fine - inizio)
                return (overlap / durata_slot) * 100

            slot_destinazione_features["slot_overlap"] = (
                slot_destinazione_features.apply(calc_overlap, axis=1).clip(upper=100.0)
            )

            slot_destinazione_features = (
                slot_destinazione_features
                .sort_values(["slot_overlap", "share_predetto"], ascending=[False, False])
                .iloc[[0]]
                .drop(columns="slot_overlap")
            )

        return (
            programma_da_spostare_features,
            slot_destinazione_features
        )
        
    
    def online_preproc(self, programma_da_spostare_features, slot_destinazione_features):
        """
        Feature Engineering, Preprocessing e creazione del record ibrido.
        """
        import pandas as pd
        
        # OHE - Programma da spostare
        for col, mapping in self.ohe_config.items():
            for original_value, feature_name in mapping.items():
                programma_da_spostare_features[feature_name] = (programma_da_spostare_features[col].astype(str) == str(original_value)).astype(int)
        # OHE - Programma nello slot di destinazione
        for col, mapping in self.ohe_config.items():
            for original_value, feature_name in mapping.items():
                slot_destinazione_features[feature_name] = (slot_destinazione_features[col].astype(str) == str(original_value)).astype(int)
        # Trasformiamo i valori delta delle feature storiche dello share in valori assoluti
        for c in ['StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast']:
            programma_da_spostare_features[c] += programma_da_spostare_features['StoricoShare']

        for c in ['StoricoShare_prev', 'StoricoShare_next', 'StoricoShareLast_prev', 'StoricoShareLast_next']:
            programma_da_spostare_features[c] += programma_da_spostare_features['StoricoShare']

        # Selezionamo solo le feature che ci servono
        programma_da_spostare_features = programma_da_spostare_features.drop(
            columns=self.programma_da_spostare_features_cols_to_drop, 
            errors='ignore'
        )
        slot_destinazione_features = slot_destinazione_features.loc[:, self.slot_destinazione_features_cols_to_keep]

        # Creiamo il dataframe di inferenza con il record ibrido
        inference_df = programma_da_spostare_features.merge(slot_destinazione_features, how="cross")
        # Ri-convertiamo i valori assoluti delle feature storiche in delta
        for c in ['StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast','StoricoShare_prev', 'StoricoShare_next', 'StoricoShareLast_prev', 'StoricoShareLast_next']:
            inference_df[c] = inference_df[c] - inference_df['StoricoShare']
        # Creazione di feature delta sull'audience
        inference_df['StoricoShareUomini_delta'] = inference_df['StoricoShareUomini'] * inference_df['StoricoShare'] - inference_df['StoricoShareUomini_competitor'] * inference_df['StoricoShare_competitor']
        inference_df['StoricoShare_08_14_delta'] = inference_df['StoricoShare_08_14'] * inference_df['StoricoShare'] - inference_df['StoricoShare_08_14_competitor'] * inference_df['StoricoShare_competitor']
        inference_df['StoricoShare_15_24_delta'] = inference_df['StoricoShare_15_24'] * inference_df['StoricoShare'] - inference_df['StoricoShare_15_24_competitor'] * inference_df['StoricoShare_competitor']
        inference_df['StoricoShare_25_64_delta'] = inference_df['StoricoShare_25_64'] * inference_df['StoricoShare'] - inference_df['StoricoShare_25_64_competitor'] * inference_df['StoricoShare_competitor']
        inference_df['StoricoShare_65_plus_delta'] = inference_df['StoricoShare_65_plus'] * inference_df['StoricoShare'] - inference_df['StoricoShare_65_plus_competitor'] * inference_df['StoricoShare_competitor']

        return inference_df
    
    def shap_waterfall_from_row(
        self,
        explainer,
        feature_row: pd.Series,
        storico_share: float,
        categorical_features: list,
        top_n_drivers = 3):
        """
        Calcola e mostra il waterfall SHAP per un singolo record.

        Richiede SOLO:
        - explainer: shap.TreeExplainer costruito dal modello (contiene gia' expected_value)
        - feature_row: la riga di feature (304 colonne) usata dal modello per la predict
        - storico_share: lo share storico del programma (in scala 0-1), necessario per
                        riportare la spiegazione in scala percentuale assoluta
        - categorical_features: lista dei nomi delle feature categoriche originali
        - top_n_drivers: gli n driver principali della predizione intesi come quelli con l'impatto POSITIVO maggiore

        """
        import shap
        from shap import Explanation
        import numpy as np
        import pandas as pd
        # ── 0. Calcolo SHAP grezzo (dal modello, nessun dataset esterno) ───────────
        row_df = feature_row.to_frame().T if isinstance(feature_row, pd.Series) else feature_row
        raw_shap = explainer(row_df)

        # ── 1. Cambio scala: residuo -> percentuale di share ──────────────────────
        # explainer.expected_value == media delle predizioni del modello sul training (proprieta' intrinseca del TreeExplainer, non serve ricalcolarla)
        base_value_pct = (explainer.expected_value + storico_share) * 100

        shap_vals = Explanation(
            values=raw_shap[0].values * 100,
            base_values=base_value_pct,
            feature_names=row_df.columns,
            data=row_df.iloc[0],
        )

        shap_series = pd.Series(shap_vals.values, index=row_df.columns)
        new_shap_values = {}
        new_data = {}
        used_cols = set()

        # ── 2. Aggregazione feature categoriche (OHE -> singola) ──────────────────
        for f in categorical_features:
            prefix = f + '_'
            if 'competitor' in f:
                cols = [c for c in row_df.columns if c.startswith(prefix)]
            else:
                cols = [c for c in row_df.columns
                        if c.startswith(prefix) and not c.startswith(prefix + 'competitor_')]

            new_shap_values[f] = shap_series[cols].sum()
            new_data[f] = pd.to_numeric(row_df.iloc[0][cols], errors='coerce').idxmax().replace(prefix, '')
            used_cols.update(cols)

        for col in row_df.columns:
            if col not in used_cols:
                new_shap_values[col] = shap_series[col]
                new_data[col] = row_df.iloc[0][col]

        # ── 3. Merge feature temporali ────────────────────────────────────────────
        new_shap_values['Ora'] = new_shap_values['Ora'] + new_shap_values.pop('ORA_INIZIO_TRX')
        new_data.pop('ORA_INIZIO_TRX')

        # ── 4. Merge statistiche StoricoShare -> assorbite nella baseline ─────────
        storico_cols = ['StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast', 'HighValueShare']
        new_shap_values['StoricoShare'] = sum(new_shap_values.pop(c) for c in storico_cols)
        for c in storico_cols:
            new_data.pop(c)
        new_data['StoricoShare'] = 'Statistiche varie'

        # ── 5. Merge StoricoShare competitor ──────────────────────────────────────
        new_shap_values['StoricoShare_competitor'] = (
            new_shap_values.pop('StoricoShareLast_competitor') + new_shap_values['StoricoShare_competitor']
        )
        new_data.pop('StoricoShareLast_competitor')
        new_data['StoricoShare_competitor'] = 'Statistiche varie'

        # ── 6. Merge feature eta' ─────────────────────────────────────────────────
        age_delta_cols = [
            'StoricoShare_08_14_delta', 'StoricoShare_15_24_delta',
            'StoricoShare_25_64_delta', 'StoricoShare_65_plus_delta',
        ]
        new_shap_values['TargetEta_vs_competitor'] = sum(new_shap_values.pop(c) for c in age_delta_cols)
        for c in age_delta_cols:
            new_data.pop(c)
        new_data['TargetEta_vs_competitor'] = 'Statistiche varie'

        # ── 7. Merge feature genere ───────────────────────────────────────────────
        new_shap_values['TargetSesso_vs_competitor'] = new_shap_values.pop('StoricoShareUomini_delta')
        new_data.pop('StoricoShareUomini_delta')
        new_data['TargetSesso_vs_competitor'] = 'Statistiche varie'

        # ── 8. Merge programmi precedente/successivo ──────────────────────────────
        new_shap_values['StoricoShare_precedente'] = (
            new_shap_values.pop('StoricoShare_prev') + new_shap_values.pop('StoricoShareLast_prev')
        )
        new_shap_values['StoricoShare_successivo'] = (
            new_shap_values.pop('StoricoShare_next') + new_shap_values.pop('StoricoShareLast_next')
        )
        for c in ['StoricoShare_prev', 'StoricoShareLast_prev', 'StoricoShare_next', 'StoricoShareLast_next']:
            new_data.pop(c, None)
        new_data['StoricoShare_precedente'] = 'Statistiche varie'
        new_data['StoricoShare_successivo'] = 'Statistiche varie'

        # ── 9. Assorbimento StoricoShare nella baseline ───────────────────────────
        storico_shap = new_shap_values.pop('StoricoShare')
        new_data.pop('StoricoShare')

        final_base = base_value_pct + storico_shap

        shap_explanation = Explanation(
            values=np.array(list(new_shap_values.values())),
            base_values=final_base,
            feature_names=np.array(list(new_shap_values.keys())),
            data=pd.Series(new_data),
        )

        pairs = list(zip(shap_explanation.feature_names, shap_explanation.values))
        top_sorted = sorted(pairs, key=lambda x: abs(x[1]), reverse=True)

        return dict(top_sorted[:top_n_drivers])

    
    def predict(self, context, model_input):
        import pandas as pd
        import shap

        # --------------------------------------------------------
        # Accept both Databricks Serving dict input and local tests
        # --------------------------------------------------------
        if isinstance(model_input, pd.DataFrame):
            # Only expected if the model was logged with a tabular signature.
            if len(model_input) != 1:
                raise ValueError("Exactly one request is expected.")

            row = model_input.iloc[0]

            programma_da_spostare = row["Programma_da_spostare"]
            destinazione = row["Destinazione"]

        elif isinstance(model_input, dict):
            # Preferred path for nested JSON
            programma_da_spostare = model_input["Programma_da_spostare"]
            destinazione = model_input["Destinazione"]

        else:
            raise TypeError(
                f"Unsupported input type: {type(model_input)}"
            )
        # --------------------------------------------------------
        # Fetch online features
        # --------------------------------------------------------
        programma_da_spostare_features, slot_destinazione_features = (
            self._fetch_features(
                programma_da_spostare=programma_da_spostare[0],
                destinazione=destinazione,
            )
        )

        inference_df = self.online_preproc(
            programma_da_spostare_features,
            slot_destinazione_features,
        )

        # --------------------------------------------------------
        # Align columns
        # --------------------------------------------------------
        X_inf = inference_df.reindex(
            columns=self.expected_columns,
            fill_value=0,
        ).fillna(0)

        type_mapping = {
            "integer": "int32",
            "long": "int64",
            "float": "float32",
            "double": "float64",
            "boolean": "bool",
        }

        for col_name, mlflow_type in self.signature_types.items():
            pandas_type = type_mapping.get(mlflow_type, "float64")

            X_inf[col_name] = pd.to_numeric(
                X_inf[col_name],
                errors="coerce",
            )

            if "int" in pandas_type:
                X_inf[col_name] = (
                    X_inf[col_name]
                    .fillna(0)
                    .astype(pandas_type)
                )
            else:
                X_inf[col_name] = (
                    X_inf[col_name]
                    .astype(pandas_type)
                )

        X_inf = X_inf.fillna(0)

        # --------------------------------------------------------
        # Prediction
        # --------------------------------------------------------
        storico_share = inference_df["StoricoShare"].iloc[0]

        prediction = self.model.predict(X_inf)[0]

        predicted_share_pct = (
            prediction + storico_share
        ) * 100

        # --------------------------------------------------------
        # SHAP
        # --------------------------------------------------------
        explainer = shap.TreeExplainer(
            self.model,
            model_output="raw",
        )

        shap_values_dict = self.shap_waterfall_from_row(
            explainer=explainer,
            feature_row=X_inf,
            storico_share=storico_share,
            categorical_features=list(self.ohe_config.keys()),
            top_n_drivers=3,
        )

        return {
            "predicted_share_pct": predicted_share_pct,
            "shap_values": shap_values_dict,
        }

### Registrazione del modello "wrapper" sull'Unity Catalog

In [0]:
programma_da_spostare_features_cols_to_drop = ['Data',
 'Canale',
'Canale_Rai_3',
 'Canale_Rai_5',
 'ORA_INIZIO_TRX',
 'ORA_FINE_TRX',
 'Programma',
 'Cod_Type_Trasm_Int',
 'Share',
 '1st_Screen_LiveVOSDAL_Bambini_06_10',
 'Tipo_Trasmissione',
 'Penetrazione',
 '1st_Screen_LiveVOSDAL_Bambini_4_7',
 'Cod_Genere_Int',
 '1st_Screen_LiveVOSDAL_Bambini_04_05',
 '1st_Screen_LiveVOSDAL_Bambini',
 '1st_Screen_LiveVOSDAL_Bambini_08_10',
 '1st_Screen_LiveVOSDAL_Ragazzi_11_14',
 '1st_Screen_LiveVOSDAL_Adulti',
 'Total_Audience',
 'LiveVOSDAL',
 '1st_Screen_LiveVOSDAL_Bambini_06_07',
 'Editore',
 'Direzione_Di_Genere',
 '1st_Screen_LiveVOSDAL_Uomini',
 '1st_Screen_LiveVOSDAL_Donne',
 '1st_Screen_LiveVOSDAL_Maschi',
 '1st_Screen_LiveVOSDAL_Femmine',
 '2nd_Screen_LiveVOSDAL_Maschi',
 '2nd_Screen_LiveVOSDAL_Femmine',
 '1st_Screen_LiveVOSDAL_Maschi_4_7',
 '1st_Screen_LiveVOSDAL_Maschi_8_14',
 '1st_Screen_LiveVOSDAL_Maschi_15_24',
 '1st_Screen_LiveVOSDAL_Maschi_25_34',
 '1st_Screen_LiveVOSDAL_Maschi_35_44',
 '1st_Screen_LiveVOSDAL_Maschi_45_54',
 '1st_Screen_LiveVOSDAL_Maschi_55_64',
 '1st_Screen_LiveVOSDAL_Maschi_65_69',
 '1st_Screen_LiveVOSDAL_Maschi_70_74',
 '1st_Screen_LiveVOSDAL_Maschi_75plus',
 '1st_Screen_LiveVOSDAL_Femmine_4_7',
 '1st_Screen_LiveVOSDAL_Femmine_8_14',
 '1st_Screen_LiveVOSDAL_Femmine_15_24',
 '1st_Screen_LiveVOSDAL_Femmine_25_34',
 '1st_Screen_LiveVOSDAL_Femmine_35_44',
 '1st_Screen_LiveVOSDAL_Femmine_45_54',
 '1st_Screen_LiveVOSDAL_Femmine_55_64',
 '1st_Screen_LiveVOSDAL_Femmine_65_69',
 '1st_Screen_LiveVOSDAL_Femmine_70_74',
 '1st_Screen_LiveVOSDAL_Femmine_75plus',
 '1st_Screen_LiveVOSDAL_Bambini_8_14',
 '1st_Screen_LiveVOSDAL_Adulti_15_24',
 '1st_Screen_LiveVOSDAL_Adulti_25_34',
 '1st_Screen_LiveVOSDAL_Adulti_35_44',
 '1st_Screen_LiveVOSDAL_Adulti_45_54',
 '1st_Screen_LiveVOSDAL_Adulti_55_64',
 '1st_Screen_LiveVOSDAL_Adulti_65_69',
 '1st_Screen_LiveVOSDAL_Adulti_70_74',
 '1st_Screen_LiveVOSDAL_Adulti_75plus',
 '1st_Screen_LiveVOSDAL',
 '2nd_Screen_LiveVOSDAL',
 '2nd_Screen_LiveVOSDAL_Eta_4_7',
 '2nd_Screen_LiveVOSDAL_Eta_8_14',
 '2nd_Screen_LiveVOSDAL_Eta_15_24',
 '2nd_Screen_LiveVOSDAL_Eta_25_34',
 '2nd_Screen_LiveVOSDAL_Eta_35_44',
 '2nd_Screen_LiveVOSDAL_Eta_45_54',
 '2nd_Screen_LiveVOSDAL_Eta_55_64',
 '2nd_Screen_LiveVOSDAL_Eta_65plus',
 '2nd_Screen_LiveVOSDAL_Altri',
 '2nd_Screen_LiveVOSDAL_Smart_Tv',
 '2nd_Screen_LiveVOSDAL_Tablet',
 '2nd_Screen_LiveVOSDAL_Smartphone',
 '2nd_Screen_LiveVOSDAL_PC',
 'COD_GEN_FILM_INT',
 'COD_GEN_SPORT_INT',
 'COD_MANIFESTAZIONE_SPORT_INT',
 'COD_NETWORK_INT',
 'COD_ORIGINE_TRX',
 'COD_SPECIALITA_SPORT_INT',
 'DES_GENERE_ESTESA_INT',
 'DES_GENERE_FILM_INT',
 'DES_GENERE_SPORT_INT',
 'DES_MANIFESTAZIONE_SPORT_INT',
 'DES_ORIGINE_TRX',
 'DES_SPECIALITA_SPORT_INT',
 'FlgPrimaSerata',
 'FlgPrimaVisione',
 'FlgPrimaVisioneGen',
 'FlgPrimaVisioneSpec',
 'Tipo_Puntata_Fiction',
 'ETA_MEDIA',
 'ID',
 'Mese',
 'GiornoSettimana',
 'Ora',
 'programma_norm',
'ORA_INIZIO_TRX','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next','StoricoShareUomini_prev','StoricoShareUomini_next','StoricoShare_competitor','StoricoShareLast_competitor','StoricoShareUomini_competitor',
                                'StoricoShare_08_14_competitor',
                                'StoricoShare_15_24_competitor',
                                'StoricoShare_25_64_competitor',
                                'StoricoShare_65_plus_competitor',
                                'Canale_Canale_5','Canale_Italia_1','Canale_Rai_1','Canale_Rai_2','Canale_Rai_3','Canale_Rete_4','Canale_Tv8','Canale_La7','Canale_Nove','Canale_competitor_20','Canale_competitor_Boing','Canale_competitor_Canale_5','Canale_competitor_Cartoonito','Canale_competitor_Cine_34','Canale_competitor_Dmax','Canale_competitor_Focus','Canale_competitor_Food_Network','Canale_competitor_Frisbee','Canale_competitor_Giallo','Canale_competitor_Iris','Canale_competitor_Italia_1','Canale_competitor_K2','Canale_competitor_La5','Canale_competitor_La7','Canale_competitor_La7_Cinema','Canale_competitor_Mediaset_Extra','Canale_competitor_Nove','Canale_competitor_Rai_1','Canale_competitor_Rai_2','Canale_competitor_Rai_3','Canale_competitor_Rai_4','Canale_competitor_Rai_5','Canale_competitor_Rai_Movie','Canale_competitor_Rai_News_24','Canale_competitor_Rai_Premium','Canale_competitor_Rai_Sport','Canale_competitor_Real_Time','Canale_competitor_Rete_4','Canale_competitor_Sky_Sport_Calcio','Canale_competitor_Sky_Sport_Uno','Canale_competitor_Sky_Tg24','Canale_competitor_Sky_Uno','Canale_competitor_Tgcom_24','Canale_competitor_Top_Crime','Canale_competitor_Tv8','Canale_competitor_Twentyseven','DES_GENERE_ESTESA_INT_competitor_ANTEPRIMA_CINEMA_TV',"DES_GENERE_ESTESA_INT_competitor_ATTUALITA'",'DES_GENERE_ESTESA_INT_competitor_Altro_editoriale_non_meglio_specificato','DES_GENERE_ESTESA_INT_competitor_BALLO','DES_GENERE_ESTESA_INT_competitor_CARTONI_ANIMATI',"DES_GENERE_ESTESA_INT_competitor_COSTUME_E_SOCIETA'",'DES_GENERE_ESTESA_INT_competitor_CUCINA','DES_GENERE_ESTESA_INT_competitor_DIBATTITO_POLITICO','DES_GENERE_ESTESA_INT_competitor_DOCUFICTION','DES_GENERE_ESTESA_INT_competitor_DOCUMENTARIO','DES_GENERE_ESTESA_INT_competitor_DOCUREALITY','DES_GENERE_ESTESA_INT_competitor_FICTION_CICLO','DES_GENERE_ESTESA_INT_competitor_FILM','DES_GENERE_ESTESA_INT_competitor_FILM_CICLO','DES_GENERE_ESTESA_INT_competitor_FILM_TV','DES_GENERE_ESTESA_INT_competitor_FOOD','DES_GENERE_ESTESA_INT_competitor_GAME_SHOW','DES_GENERE_ESTESA_INT_competitor_INCHIESTE','DES_GENERE_ESTESA_INT_competitor_INFORMAZIONE_PARLAMENTARE','DES_GENERE_ESTESA_INT_competitor_LIFESTYLE_ALTRO','DES_GENERE_ESTESA_INT_competitor_LIRICA','DES_GENERE_ESTESA_INT_competitor_MANIFESTAZIONI','DES_GENERE_ESTESA_INT_competitor_MINISERIE','DES_GENERE_ESTESA_INT_competitor_MONOGRAFIE','DES_GENERE_ESTESA_INT_competitor_MUSICALE','DES_GENERE_ESTESA_INT_competitor_PREVISIONI_DEL_TEMPO','DES_GENERE_ESTESA_INT_competitor_PROGRAMMA_DI_MONTAGGIO','DES_GENERE_ESTESA_INT_competitor_PROGRAMMA_PER_BAMBINI/RAGAZZI','DES_GENERE_ESTESA_INT_competitor_PROGRAMMAZIONI_CINEMATOGRAFICHE','DES_GENERE_ESTESA_INT_competitor_PROGRAMMI_DIBATTITO/INFORMATIVO','DES_GENERE_ESTESA_INT_competitor_PROPERTY','DES_GENERE_ESTESA_INT_competitor_PROSSIMAMENTE','DES_GENERE_ESTESA_INT_competitor_REALITY_SHOW','DES_GENERE_ESTESA_INT_competitor_REDAZIONALE','DES_GENERE_ESTESA_INT_competitor_ROTOCALCO_LEGGERO','DES_GENERE_ESTESA_INT_competitor_RUBRICA','DES_GENERE_ESTESA_INT_competitor_RUBRICA_RELIGIOSA','DES_GENERE_ESTESA_INT_competitor_RUBRICA_SPORTIVA','DES_GENERE_ESTESA_INT_competitor_RUBRICHE_DEL_TELEGIORNALE','DES_GENERE_ESTESA_INT_competitor_RUBRICHE_DI_SERVIZIO','DES_GENERE_ESTESA_INT_competitor_RUBRICHE_SCOLASTICHE','DES_GENERE_ESTESA_INT_competitor_SANTA_MESSA','DES_GENERE_ESTESA_INT_competitor_SCIENZA_ED_AMBIENTE','DES_GENERE_ESTESA_INT_competitor_SEGNALE_ORARIO','DES_GENERE_ESTESA_INT_competitor_SITUATION_COMEDY','DES_GENERE_ESTESA_INT_competitor_SOAP_OPERA_/_TELEROMANZO','DES_GENERE_ESTESA_INT_competitor_SPETTACOLO_MUSICALE','DES_GENERE_ESTESA_INT_competitor_SPORT','DES_GENERE_ESTESA_INT_competitor_STORIA/LETTERATURA/ARTE','DES_GENERE_ESTESA_INT_competitor_TALENT_ALTRO','DES_GENERE_ESTESA_INT_competitor_TALK_SHOW','DES_GENERE_ESTESA_INT_competitor_TELEFILM','DES_GENERE_ESTESA_INT_competitor_TELEGIORNALE','DES_GENERE_ESTESA_INT_competitor_TELENOVELA','DES_GENERE_ESTESA_INT_competitor_TG_SPORT','DES_GENERE_ESTESA_INT_competitor_TRASMISSIONE_A_QUIZ','DES_GENERE_ESTESA_INT_competitor_TRASMISSIONI_DI_SERVIZIO','DES_GENERE_ESTESA_INT_competitor_TRIBUNA_POLITICA','DES_GENERE_ESTESA_INT_competitor_TUTORIAL',"DES_GENERE_ESTESA_INT_competitor_VARIETA'",'DES_GENERE_ESTESA_INT_competitor_VIAGGI','DES_GENERE_ESTESA_INT_competitor_WEDDING', 'Ora_10_0','Ora_11_0','Ora_12_0','Ora_13_0','Ora_14_0','Ora_15_0','Ora_16_0','Ora_17_0','Ora_18_0','Ora_19_0','Ora_20_0','Ora_21_0','Ora_22_0','Ora_23_0','Ora_24_0','Ora_25_0','Ora_26_0','Ora_7_0','Ora_8_0','Ora_9_0','GiornoSettimana_0','GiornoSettimana_1','GiornoSettimana_2','GiornoSettimana_3','GiornoSettimana_4','GiornoSettimana_5','GiornoSettimana_6','Mese_1','Mese_10','Mese_11','Mese_12','Mese_2','Mese_3','Mese_4','Mese_5','Mese_6','Mese_7','Mese_8','Mese_9'
]

slot_destinazione_features_cols_to_keep = ['ORA_INIZIO_TRX','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next','StoricoShareUomini_prev','StoricoShareUomini_next','StoricoShare_competitor','StoricoShareLast_competitor','StoricoShareUomini_competitor',
                                'StoricoShare_08_14_competitor',
                                'StoricoShare_15_24_competitor',
                                'StoricoShare_25_64_competitor',
                                'StoricoShare_65_plus_competitor',
                                'Canale_Canale_5','Canale_Italia_1','Canale_Rai_1','Canale_Rai_2','Canale_Rai_3','Canale_Rete_4','Canale_Tv8','Canale_La7','Canale_Nove','Canale_competitor_20','Canale_competitor_Boing','Canale_competitor_Canale_5','Canale_competitor_Cartoonito','Canale_competitor_Cine_34','Canale_competitor_Dmax','Canale_competitor_Focus','Canale_competitor_Food_Network','Canale_competitor_Frisbee','Canale_competitor_Giallo','Canale_competitor_Iris','Canale_competitor_Italia_1','Canale_competitor_K2','Canale_competitor_La5','Canale_competitor_La7','Canale_competitor_La7_Cinema','Canale_competitor_Mediaset_Extra','Canale_competitor_Nove','Canale_competitor_Rai_1','Canale_competitor_Rai_2','Canale_competitor_Rai_3','Canale_competitor_Rai_4','Canale_competitor_Rai_5','Canale_competitor_Rai_Movie','Canale_competitor_Rai_News_24','Canale_competitor_Rai_Premium','Canale_competitor_Rai_Sport','Canale_competitor_Real_Time','Canale_competitor_Rete_4','Canale_competitor_Sky_Sport_Calcio','Canale_competitor_Sky_Sport_Uno','Canale_competitor_Sky_Tg24','Canale_competitor_Sky_Uno','Canale_competitor_Tgcom_24','Canale_competitor_Top_Crime','Canale_competitor_Tv8','Canale_competitor_Twentyseven','DES_GENERE_ESTESA_INT_competitor_ANTEPRIMA_CINEMA_TV',"DES_GENERE_ESTESA_INT_competitor_ATTUALITA'",'DES_GENERE_ESTESA_INT_competitor_Altro_editoriale_non_meglio_specificato','DES_GENERE_ESTESA_INT_competitor_BALLO','DES_GENERE_ESTESA_INT_competitor_CARTONI_ANIMATI',"DES_GENERE_ESTESA_INT_competitor_COSTUME_E_SOCIETA'",'DES_GENERE_ESTESA_INT_competitor_CUCINA','DES_GENERE_ESTESA_INT_competitor_DIBATTITO_POLITICO','DES_GENERE_ESTESA_INT_competitor_DOCUFICTION','DES_GENERE_ESTESA_INT_competitor_DOCUMENTARIO','DES_GENERE_ESTESA_INT_competitor_DOCUREALITY','DES_GENERE_ESTESA_INT_competitor_FICTION_CICLO','DES_GENERE_ESTESA_INT_competitor_FILM','DES_GENERE_ESTESA_INT_competitor_FILM_CICLO','DES_GENERE_ESTESA_INT_competitor_FILM_TV','DES_GENERE_ESTESA_INT_competitor_FOOD','DES_GENERE_ESTESA_INT_competitor_GAME_SHOW','DES_GENERE_ESTESA_INT_competitor_INCHIESTE','DES_GENERE_ESTESA_INT_competitor_INFORMAZIONE_PARLAMENTARE','DES_GENERE_ESTESA_INT_competitor_LIFESTYLE_ALTRO','DES_GENERE_ESTESA_INT_competitor_LIRICA','DES_GENERE_ESTESA_INT_competitor_MANIFESTAZIONI','DES_GENERE_ESTESA_INT_competitor_MINISERIE','DES_GENERE_ESTESA_INT_competitor_MONOGRAFIE','DES_GENERE_ESTESA_INT_competitor_MUSICALE','DES_GENERE_ESTESA_INT_competitor_PREVISIONI_DEL_TEMPO','DES_GENERE_ESTESA_INT_competitor_PROGRAMMA_DI_MONTAGGIO','DES_GENERE_ESTESA_INT_competitor_PROGRAMMA_PER_BAMBINI/RAGAZZI','DES_GENERE_ESTESA_INT_competitor_PROGRAMMAZIONI_CINEMATOGRAFICHE','DES_GENERE_ESTESA_INT_competitor_PROGRAMMI_DIBATTITO/INFORMATIVO','DES_GENERE_ESTESA_INT_competitor_PROPERTY','DES_GENERE_ESTESA_INT_competitor_PROSSIMAMENTE','DES_GENERE_ESTESA_INT_competitor_REALITY_SHOW','DES_GENERE_ESTESA_INT_competitor_REDAZIONALE','DES_GENERE_ESTESA_INT_competitor_ROTOCALCO_LEGGERO','DES_GENERE_ESTESA_INT_competitor_RUBRICA','DES_GENERE_ESTESA_INT_competitor_RUBRICA_RELIGIOSA','DES_GENERE_ESTESA_INT_competitor_RUBRICA_SPORTIVA','DES_GENERE_ESTESA_INT_competitor_RUBRICHE_DEL_TELEGIORNALE','DES_GENERE_ESTESA_INT_competitor_RUBRICHE_DI_SERVIZIO','DES_GENERE_ESTESA_INT_competitor_RUBRICHE_SCOLASTICHE','DES_GENERE_ESTESA_INT_competitor_SANTA_MESSA','DES_GENERE_ESTESA_INT_competitor_SCIENZA_ED_AMBIENTE','DES_GENERE_ESTESA_INT_competitor_SEGNALE_ORARIO','DES_GENERE_ESTESA_INT_competitor_SITUATION_COMEDY','DES_GENERE_ESTESA_INT_competitor_SOAP_OPERA_/_TELEROMANZO','DES_GENERE_ESTESA_INT_competitor_SPETTACOLO_MUSICALE','DES_GENERE_ESTESA_INT_competitor_SPORT','DES_GENERE_ESTESA_INT_competitor_STORIA/LETTERATURA/ARTE','DES_GENERE_ESTESA_INT_competitor_TALENT_ALTRO','DES_GENERE_ESTESA_INT_competitor_TALK_SHOW','DES_GENERE_ESTESA_INT_competitor_TELEFILM','DES_GENERE_ESTESA_INT_competitor_TELEGIORNALE','DES_GENERE_ESTESA_INT_competitor_TELENOVELA','DES_GENERE_ESTESA_INT_competitor_TG_SPORT','DES_GENERE_ESTESA_INT_competitor_TRASMISSIONE_A_QUIZ','DES_GENERE_ESTESA_INT_competitor_TRASMISSIONI_DI_SERVIZIO','DES_GENERE_ESTESA_INT_competitor_TRIBUNA_POLITICA','DES_GENERE_ESTESA_INT_competitor_TUTORIAL',"DES_GENERE_ESTESA_INT_competitor_VARIETA'",'DES_GENERE_ESTESA_INT_competitor_VIAGGI','DES_GENERE_ESTESA_INT_competitor_WEDDING', 'Ora_10_0','Ora_11_0','Ora_12_0','Ora_13_0','Ora_14_0','Ora_15_0','Ora_16_0','Ora_17_0','Ora_18_0','Ora_19_0','Ora_20_0','Ora_21_0','Ora_22_0','Ora_23_0','Ora_24_0','Ora_25_0','Ora_26_0','Ora_7_0','Ora_8_0','Ora_9_0','GiornoSettimana_0','GiornoSettimana_1','GiornoSettimana_2','GiornoSettimana_3','GiornoSettimana_4','GiornoSettimana_5','GiornoSettimana_6','Mese_1','Mese_10','Mese_11','Mese_12','Mese_2','Mese_3','Mese_4','Mese_5','Mese_6','Mese_7','Mese_8','Mese_9'
]


In [0]:
import mlflow
from mlflow.models import infer_signature
with mlflow.start_run() as run:

    input_example = {
		"Programma_da_spostare": ["Rai 1_2026-07-14_capri_14:05"],
		"Destinazione":
		{
			"Canale": ["Rai 2"],
			"data_str": ["2026-07-15"],
			"orario": ["15:00"],
			"programmi_contesto": ["Rai 3_2026-07-15_tg3 lis_15:00", "Rai 3_2026-07-15_rai parlamento telegiornale_15:05", "Rai 3_2026-07-15_il provinciale_15:10", "Rai 3_2026-07-15_di la dal fiume e tra gli alberi cuneo_15:45", "Rai 3_2026-07-15_overland 20 africa dellovest_16:45"]
		}
	}


    # Example output — whatever your model.predict() returns
    output_example = {
        "predicted_share_pct": 2.83,
        "shap_values": {"Canale" : 2.40, "Programma_Competitor": 3.00, "orario_inizio": 1.05}
    }

    signature = infer_signature(input_example, output_example)

    mlflow.pyfunc.log_model(
        artifact_path="whatif_spostamento_wrapper_model",
        python_model=WhatIfSostituzionModel(),
        artifacts={
            "inference_model": model_uri,
            "ohe_json": "../FASE1/categorical_features_ohe.json"
        },
        extra_pip_requirements=[
            "databricks-feature-engineering",
            "mlflow==2.21.3",
            "cloudpickle==2.2.1",
            "lz4==4.3.2",
            "numpy<2.0",
            "pandas",
            "psutil",
            "scikit-learn==1.4.2",
            "xgboost==2.0.3",
            "shap"
        ],
        model_config={
            "endpoint_feature_all_slots": "features_all_slots_preds",
            "programma_da_spostare_features_cols_to_drop": programma_da_spostare_features_cols_to_drop,
            "slot_destinazione_features_cols_to_keep": slot_destinazione_features_cols_to_keep
        },
        signature=signature,
        input_example=input_example,
    )